# 원인 분석 — Dimension #2: 집객시설 특성

[기획안_설계.md](기획안_설계.md) 6번 화면2 Section 1의 Dimension #2를 만드는 노트북.

**질문**: 위험 상권(299개, [02_EDA_상권쇠퇴분석.ipynb](02_EDA_상권쇠퇴분석.ipynb) STEP 2에서 도출)은 비위험 상권과 비교했을 때 지하철역·학교·병원 같은 집객시설(인프라)이 더 많은가, 적은가?

**비교 방법**: 위험 상권 그룹과 비위험 상권 그룹의 상권당 평균 시설 수를 비교한다. 집객시설은 연 1회(4분기) 갱신 스냅샷이라 [데이터_명세.md](데이터_명세.md)에 따라 **정적 상관관계로만** 활용 — 시계열 결합 금지.

---
## STEP 1. 위험 상권 재현

이 노트북은 새 파일이라 이전 노트북의 변수를 이어받을 수 없음. [03_원인분석_업종구성.ipynb](03_원인분석_업종구성.ipynb) STEP 1과 동일한 코드로 위험 상권(299개) 집합만 다시 만든다.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

DATA_DIR = "../data"
df_stores = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_stores_2025.csv", encoding="cp949")
df_master = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_master_2025.csv", encoding="cp949")

print("stores:", df_stores.shape)
print("master:", df_master.shape)

stores: (304775, 12)
master: (1650, 11)


In [2]:
# --- 컬럼 보정 (02번 노트북 STEP 1과 동일 로직) ---
df_stores_fixed = df_stores.copy()

# 1) 프랜차이즈_점포_수 보정: round(일반 * 프차 / 100)
전체_보정후_예상값 = (df_stores_fixed["일반_점포_수"] * df_stores_fixed["프랜차이즈_점포_수"] / 100).round(0).astype(int)
원래_공식_성립 = df_stores_fixed["전체_점포_수"] == (df_stores_fixed["일반_점포_수"] + df_stores_fixed["프랜차이즈_점포_수"])
df_stores_fixed["프랜차이즈_점포_수_보정"] = np.where(원래_공식_성립, df_stores_fixed["프랜차이즈_점포_수"], 전체_보정후_예상값)

# 2) 일반_점포_수 보정: 전체 - 프랜차이즈(보정), 음수는 0으로 클리핑
df_stores_fixed["일반_점포_수_보정"] = (df_stores_fixed["전체_점포_수"] - df_stores_fixed["프랜차이즈_점포_수_보정"]).clip(lower=0)

# 3) 전체_점포_수 보정: max(원본, 일반보정+프차보정)
전체_후보 = df_stores_fixed["일반_점포_수_보정"] + df_stores_fixed["프랜차이즈_점포_수_보정"]
df_stores_fixed["전체_점포_수_보정"] = np.maximum(df_stores_fixed["전체_점포_수"], 전체_후보)

# 4) 개업_점포_수 보정: 전체_점포_수_보정을 넘는 값만 클리핑
df_stores_fixed["개업_점포_수_보정"] = np.minimum(df_stores_fixed["개업_점포_수"], df_stores_fixed["전체_점포_수_보정"])

# 검증: 공식이 100% 맞는지 재확인
mismatch = df_stores_fixed[df_stores_fixed["전체_점포_수_보정"] != (df_stores_fixed["일반_점포_수_보정"] + df_stores_fixed["프랜차이즈_점포_수_보정"])]
print(f"보정 후 공식이 안 맞는 행: {len(mismatch)} / {len(df_stores_fixed)} (0이어야 함)")

보정 후 공식이 안 맞는 행: 0 / 304775 (0이어야 함)


In [3]:
# --- 상권×분기 집계 ---
group_keys = ["상권_코드", "상권_코드_명", "상권_구분_코드_명", "기준_년분기_코드"]
sum_cols_fixed = ["전체_점포_수_보정", "일반_점포_수_보정", "프랜차이즈_점포_수_보정", "개업_점포_수_보정", "폐업_점포_수"]

df_agg = df_stores_fixed.groupby(group_keys, as_index=False)[sum_cols_fixed].sum()
df_agg = df_agg.rename(columns={
    "전체_점포_수_보정": "전체_점포_수",
    "일반_점포_수_보정": "일반_점포_수",
    "프랜차이즈_점포_수_보정": "프랜차이즈_점포_수",
    "개업_점포_수_보정": "개업_점포_수",
})

print("상권×분기 집계:", df_agg.shape, "(02번 노트북과 같은 (6600, 9)여야 함)")

상권×분기 집계: (6600, 9) (02번 노트북과 같은 (6600, 9)여야 함)


In [4]:
# --- KPI-1(순증감율), KPI-2(폐업률 증감) 계산 후 위험 상권 판정 ---
분모_안전 = df_agg["전체_점포_수"].replace(0, np.nan)
df_agg["KPI1_순증감율"] = (df_agg["개업_점포_수"] - df_agg["폐업_점포_수"]) / 분모_안전
df_agg["폐업률"] = df_agg["폐업_점포_수"] / 분모_안전

df_agg = df_agg.sort_values(["상권_코드", "기준_년분기_코드"]).reset_index(drop=True)
df_agg["폐업률_전분기"] = df_agg.groupby("상권_코드")["폐업률"].shift(1)
df_agg["KPI2_폐업률증감"] = df_agg["폐업률"] - df_agg["폐업률_전분기"]

# 최신 분기(20254) 기준 위험 판정: 순증감율<0 AND 폐업률증감>0
df_최신분기 = df_agg[df_agg["기준_년분기_코드"] == 20254].copy()
위험_조건 = (df_최신분기["KPI1_순증감율"] < 0) & (df_최신분기["KPI2_폐업률증감"] > 0)
df_위험상권 = df_최신분기[위험_조건]

print(f"위험 상권 수: {len(df_위험상권)} (02번 노트북과 같은 299개여야 함, 전체 {len(df_최신분기)}개 중)")

위험 상권 수: 299 (02번 노트북과 같은 299개여야 함, 전체 1650개 중)


---
## STEP 2. 집객시설 데이터 로드 + 위험상권 여부 태깅

`attraction_facilities` 파일은 [데이터_명세.md](데이터_명세.md)에 따라 인코딩이 **utf-8-sig**로 다른 두 파일(cp949)과 다름. 또한 연 1회(4분기) 갱신이라 1~3분기 값이 4분기와 동일하게 복제돼있으므로, **20254(최신 분기) 한 줄만** 골라 쓴다.

하는 일:
1. 집객시설 CSV 로드 (encoding="utf-8-sig")
2. 20254 분기만 추출
3. 세부 시설 컬럼의 결측치(NaN) → 0으로 치환 ("해당 시설 없음"으로 해석, 명세서 규칙)
4. STEP 1에서 만든 위험 상권 집합으로 `위험상권_여부` 컬럼 추가

In [5]:
df_facilities = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_attraction_facilities_2025.csv", encoding="utf-8-sig")
print("집객시설 원본:", df_facilities.shape)

# 20254(최신 분기)만 추출 — 1~3분기는 4분기 값의 복제본이라 중복
df_최신분기_시설 = df_facilities[df_facilities["기준_년분기_코드"] == 20254].copy()
print("20254 분기만:", df_최신분기_시설.shape, "(1,650개 상권 중 일부만 존재할 수 있음)")

# 세부 시설 컬럼 목록 (기준/코드/이름 컬럼 제외한 수치 컬럼 전부)
메타_컬럼 = ["기준_년분기_코드", "상권_구분_코드", "상권_구분_코드_명", "상권_코드", "상권_코드_명"]
시설_컬럼 = [c for c in df_최신분기_시설.columns if c not in 메타_컬럼]

# 결측치(NaN) -> 0으로 치환
print("치환 전 결측치 개수 합:", df_최신분기_시설[시설_컬럼].isna().sum().sum())
df_최신분기_시설[시설_컬럼] = df_최신분기_시설[시설_컬럼].fillna(0)
print("치환 후 결측치 개수 합:", df_최신분기_시설[시설_컬럼].isna().sum().sum(), "(0이어야 함)")

집객시설 원본: (6312, 25)
20254 분기만: (1578, 25) (1,650개 상권 중 일부만 존재할 수 있음)
치환 전 결측치 개수 합: 25625
치환 후 결측치 개수 합: 0 (0이어야 함)


In [6]:
# 위험 상권 여부 태깅
위험_상권코드_집합 = set(df_위험상권["상권_코드"])
df_최신분기_시설["위험상권_여부"] = df_최신분기_시설["상권_코드"].isin(위험_상권코드_집합)

# 검증: 위험상권_여부=True인 행의 고유 상권 수가 299개와 같아야 함 (단, 시설 데이터에 없는 위험상권이 있으면 이보다 작을 수 있음)
매칭된_위험상권_수 = df_최신분기_시설.loc[df_최신분기_시설["위험상권_여부"], "상권_코드"].nunique()
print(f"집객시설 데이터에 있는 위험 상권 수: {매칭된_위험상권_수} / 299")
print(f"집객시설 데이터 자체에 없는 위험 상권 수: {299 - 매칭된_위험상권_수}개 (master 1,650개 중 일부가 시설 데이터에서 빠져있는 것과 같은 이유)")

집객시설 데이터에 있는 위험 상권 수: 294 / 299
집객시설 데이터 자체에 없는 위험 상권 수: 5개 (master 1,650개 중 일부가 시설 데이터에서 빠져있는 것과 같은 이유)


---
## STEP 3. 위험 상권 vs 비위험 상권, 상권당 평균 시설 수 비교

업종 분석(03번 노트북)과 달리 집객시설은 상권마다 하나의 값(상권당 개수)이므로, 점포 수 비중이 아니라 **그룹별 평균**으로 비교하는 게 자연스럽다.

하는 일:
1. `위험상권_여부`로 그룹을 나눠 시설 컬럼별 평균 계산
2. 위험 그룹 평균 − 비위험 그룹 평균(차이) 계산, 차이가 큰 순으로 정렬

In [7]:
# groupby(위험상권_여부) 후 시설 컬럼들의 평균을 계산 -> 행 2개(True/False), 열은 시설 종류
그룹별_평균 = df_최신분기_시설.groupby("위험상권_여부")[시설_컬럼].mean().T
그룹별_평균.columns = ["비위험상권_평균", "위험상권_평균"]

그룹별_평균["차이(위험-비위험)"] = 그룹별_평균["위험상권_평균"] - 그룹별_평균["비위험상권_평균"]

print("=== 위험 상권에 시설이 더 많은 순 (Top 10) ===")
print(그룹별_평균.sort_values("차이(위험-비위험)", ascending=False).head(10).round(2))

print()
print("=== 위험 상권에 시설이 더 적은 순 (Top 10) ===")
print(그룹별_평균.sort_values("차이(위험-비위험)", ascending=True).head(10).round(2))

=== 위험 상권에 시설이 더 많은 순 (Top 10) ===
          비위험상권_평균  위험상권_평균  차이(위험-비위험)
집객시설_수       20.34    23.20        2.87
약국_수          2.29     2.67        0.38
버스_정거장_수      3.85     4.12        0.27
은행_수          0.99     1.14        0.15
지하철_역_수       0.16     0.23        0.07
일반_병원_수       0.10     0.14        0.04
숙박_시설_수       0.21     0.22        0.01
공항_수          0.00     0.01        0.01
극장_수          0.07     0.08        0.01
종합병원_수        0.01     0.02        0.01

=== 위험 상권에 시설이 더 적은 순 (Top 10) ===
          비위험상권_평균  위험상권_평균  차이(위험-비위험)
대학교_수         0.09     0.07       -0.02
슈퍼마켓_수        0.10     0.10       -0.01
고등학교_수        0.01     0.00       -0.01
관공서_수         0.70     0.69       -0.01
백화점_수         0.01     0.00       -0.01
철도_역_수        0.00     0.00        0.00
유치원_수         0.11     0.11        0.00
중학교_수         0.00     0.00        0.00
초등학교_수        0.01     0.01        0.00
버스_터미널_수      0.00     0.00        0.00


### 해석 결론

- **위험 상권은 집객시설 총량 자체가 더 많음** (평균 23.2개 vs 20.3개) — 인프라가 부족해서 쇠퇴하는 게 아니라 오히려 시설이 더 밀집된 곳에서 쇠퇴가 나타남.
- 그중 **약국·버스정거장·은행·지하철역**이 위험 상권에 더 많음 — 유동인구/교통 접근성은 있는데 상권 활력은 떨어지는 "인프라 대비 상권 활력 불균형" 패턴으로 해석 가능.
- 대학교·백화점·고등학교는 비위험 상권에 근소하게 많지만 차이가 0.01~0.02개 수준으로 작아 실질적 의미는 약함.
- **주의**: 종합병원_수·중학교_수·초등학교_수·버스_터미널_수처럼 상권당 평균이 0.00~0.01개로 극히 작은 시설은 표본이 너무 적어 배율(비율) 비교가 착시를 일으킴 — 이 노트북에서는 배율 대신 절대 차이만 비교 지표로 채택.